# Permutation test v2 — tín hiệu giai đoạn trên mẫu u (RQ0)

Thực hiện kịch bản `docs/kich-ban-permutation-test-v2.docx`. Câu hỏi: **trên mẫu u (bỏ Normal), biểu hiện gen có
mang tín hiệu phân biệt giai đoạn hay không, và chọn gen có khai thác được tín hiệu đó tốt hơn chọn gen ngẫu nhiên không?**

- **Dữ liệu:** gộp `train/val/test` của `fold_1` (= toàn bộ bộ dữ liệu), **bỏ mẫu Normal**, giữ nguyên log2 (không chuẩn hoá).
- **Nhiệm vụ:** `stage_all` (I–IV; GSE68465: I–III), `early_late` (I vs II–IV), `stage_merge_34` (I/II/III+IV, chỉ TCGA).
- **3 pipeline (chọn gen + phân loại):** P1 F-test + LR (Zhu & Hastie, 2004; như notebook v1) ·
  P2 SVM-RFE + SVM tuyến tính (Guyon et al., 2002) · P3 Nearest Shrunken Centroids/PAM (Tibshirani et al., 2002).
- **Số gene:** k ∈ {10, 50, 100, 200, 1000}. **class_weight = None** (không trọng số lớp) cho mọi mô hình.
- **Chia fold:** `StratifiedGroupKFold(5)` theo bệnh nhân, cố định, dùng chung cho mọi pipeline và mọi hoán vị.
  Bước chọn gen nằm **trong** từng fold và được chạy lại ở mọi hoán vị.
- **Permutation:** 1000 lần xáo nhãn → p = (#null ≥ thật + 1)/(1000 + 1). Hoán vị thứ *i* luôn dùng cùng seed
  → các phiên chạy khác nhau cho cùng một bộ nhãn xáo (cho phép chia pipeline ra nhiều phiên).
- **Hiệu chỉnh đa kiểm định:** họ = 3 pipeline × 5 k = 15 kiểm định cho mỗi (bộ dữ liệu, nhiệm vụ);
  báo cáo p gốc, **Holm**, **Bonferroni** và Westfall–Young maxT (tham khảo).
- **Random-k:** 200 lần chọn ngẫu nhiên k gene + cùng bộ phân loại của từng pipeline.
- **Chạy nhiều phiên / tiếp tục:** notebook tự dừng trước `TIME_BUDGET_H` và lưu tiến độ. Phiên sau: thêm output
  phiên trước làm Input, điền đường dẫn vào `RESUME_DIRS`, chạy lại — phần đã xong được bỏ qua.
- **Đầu ra** (`/kaggle/working/perm_test_v2/<dataset>/` → `perm_test_v2_<dataset>.zip`): `summary.csv`, `decision.csv`,
  `real_scores_per_fold.csv`, `real_confusion.csv`, `null_scores.csv`, `random_k_scores.csv`, `timing.csv`,
  `perm_test_v2_bacc.png`, `perm_test_v2_pvalues.png`, `config.json`.

In [ ]:
# tables: backend cho pd.read_hdf
get_ipython().system("pip install -q tables")

## 1. Cấu hình

In [ ]:
import os

# ====== CAU HINH (sua tay) ======
# Doi DATA_DIR sang bo khac (.../h5/tcga_luad, .../h5/tcga_lusc) de chay dataset khac
DATA_DIR = "/kaggle/input/datasets/lfreedom2750/experiment-dataset/h5/gse68465"
# Pipeline chay trong phien nay (ca 3 pipeline xong trong 1 phien: ~2-3 gio TCGA, ~1 gio GSE68465)
PIPELINES = ["ftest_lr", "svm_rfe", "pam"]
ALL_PIPELINES = ["ftest_lr", "svm_rfe", "pam"]   # ho kiem dinh day du (de hieu chinh p)
K_LIST = [10, 50, 100, 200, 1000]
N_PERMUTATIONS = 1000
N_RANDOM_K = 200
N_SPLITS = 5
SEED = 42
LR_C = 1.0                # LR phat L2 cua P1
SVM_C = 1.0               # SVM tuyen tinh cua P2
ALPHA = 0.05
MIN_GAIN = 0.02           # muc chenh BAcc toi thieu so voi random-k de coi la "co ich" (dat truoc)
N_JOBS = -1
BATCH_PER_WORKER = 4      # so hoan vi moi worker trong 1 lo; luu ket qua sau moi lo
TIME_BUDGET_H = 11.0      # dung an toan truoc gioi han 12h cua Kaggle
# Phien tiep theo: them output phien truoc lam Input roi dien thu muc chua null_scores.csv, vd
# ["/kaggle/input/perm-v2-gse68465-part1/perm_test_v2/gse68465"]
RESUME_DIRS = []

DATASET_NAME = os.path.basename(os.path.normpath(DATA_DIR))
RESULT_DIR = os.path.join("/kaggle/working/perm_test_v2", DATASET_NAME)
os.makedirs(RESULT_DIR, exist_ok=True)
N_WORKERS = os.cpu_count() if N_JOBS == -1 else N_JOBS

assert os.path.isfile(os.path.join(DATA_DIR, "fold_1", "train.h5")), \
    f"Khong thay {DATA_DIR}/fold_1/train.h5 -- da Add Input dataset chua?"
assert set(PIPELINES) <= set(ALL_PIPELINES)
print("Dataset  :", DATASET_NAME, "->", DATA_DIR)
print("Pipeline :", PIPELINES, "| k =", K_LIST)
print("Hoan vi  :", N_PERMUTATIONS, "| random-k:", N_RANDOM_K, "| ngan sach:", TIME_BUDGET_H, "gio")
print("Ket qua  :", RESULT_DIR, "| so CPU:", os.cpu_count())

## 2. Dữ liệu (chỉ mẫu u), nhiệm vụ và chia fold theo bệnh nhân

In [ ]:
import json, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")
T_START = time.time()
STAGE_NAMES = {0: "Normal", 1: "I", 2: "II", 3: "III", 4: "IV"}

parts = [pd.read_hdf(os.path.join(DATA_DIR, "fold_1", f"{s}.h5"), key="data") for s in ["train", "val", "test"]]
df = pd.concat(parts)
assert df.index.is_unique, "Trung sample_id giua train/val/test"
print(f"Toan bo: {df.shape[0]} mau x {df.shape[1] - 1} gene |",
      df["label"].map(STAGE_NAMES).value_counts().to_dict())

df = df[df["label"] != 0]                                     # BO mau Normal
X = np.ascontiguousarray(df.drop(columns="label").values, dtype=np.float64)   # log2 goc, KHONG chuan hoa
y_stage = df["label"].values.astype(int)
sample_ids = df.index.to_numpy().astype(str)
groups = np.array([s[:12] if s.startswith("TCGA-") else s for s in sample_ids])   # ma benh nhan
n_dup = int((pd.Series(groups).value_counts() > 1).sum())
del df, parts
print(f"Mau u: {X.shape[0]} x {X.shape[1]} gene | benh nhan co >1 mau: {n_dup}")
print("Giai doan:", pd.Series(y_stage).map(STAGE_NAMES).value_counts().sort_index().to_dict())

TASK_LABELS = {
    "stage_all": y_stage,
    "early_late": (y_stage >= 2).astype(int),          # 0 = som (I), 1 = muon (II-IV)
    "stage_merge_34": np.minimum(y_stage, 3),           # gop IV vao III
}
if 4 not in y_stage:                                    # GSE68465: khong co IV -> merge_34 trung stage_all
    del TASK_LABELS["stage_merge_34"]
TASKS = list(TASK_LABELS)
TASK_CHANCE = {t: 1 / len(np.unique(v)) for t, v in TASK_LABELS.items()}
for t in TASKS:
    print(f"  {t:15s} lop {np.unique(TASK_LABELS[t]).tolist()} | muc ngau nhien BAcc {TASK_CHANCE[t]:.3f}")

# Fold co dinh (phan tang theo giai doan that, theo benh nhan), dung chung cho moi nhiem vu/pipeline/hoan vi
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
FOLDS = list(sgkf.split(X, y_stage, groups))
for i, (tr, te) in enumerate(FOLDS, 1):
    assert set(groups[tr]).isdisjoint(groups[te]), f"fold {i}: trung benh nhan train/test"
# Tach san du lieu tung fold -> joblib memmap, worker khong phai copy lai
FOLD_DATA = [(X[tr], X[te]) for tr, te in FOLDS]
print("Kich thuoc test moi fold:", [len(te) for _, te in FOLDS])

## 3. Ba pipeline chọn gen + phân loại

| Mã | Chọn gen (trong fold) | Phân loại | Tham số cố định | Cơ sở |
|---|---|---|---|---|
| `ftest_lr` | F-test ANOVA (xếp hạng đơn biến), top-k | LR phạt L2 | C = 1 | Zhu & Hastie (2004); Liao & Chin (2007); notebook v1 |
| `svm_rfe` | SVM-RFE: loại một nửa số gene mỗi vòng theo w², dừng đúng tại từng k | SVM tuyến tính (one-vs-rest khi > 2 lớp) | C = 1 | Guyon, Weston, Barnhill & Vapnik (2002) |
| `pam` | Nearest shrunken centroids: ngưỡng Δ đặt sao cho còn đúng k gene | Shrunken centroid (prior = tỉ lệ lớp) | s0 = trung vị s_i | Tibshirani, Hastie, Narasimhan & Chu (2002) |

Không pipeline nào dùng trọng số lớp (`class_weight = None`; PAM dùng prior bằng tỉ lệ lớp). SVM dùng kernel tính sẵn
`X·Xᵀ` (tương đương `SVC(kernel="linear")`, nhanh hơn nhiều khi n ≪ p). Với `ftest_lr` và `pam`, tập k gene là tiền tố
của **một** xếp hạng mỗi fold; SVM-RFE dừng lại ghi nhận ở từng k.

In [ ]:
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score


# ---------- P1: F-test + LR (notebook v1) ----------
def ftest_rank(X, y):
    F, _ = f_classif(X, y)                                   # gene hang so -> NaN -> xuong cuoi
    return np.argsort(-np.nan_to_num(F, nan=-np.inf), kind="stable")


def make_lr():
    return LogisticRegression(C=LR_C, max_iter=5000)        # L2, class_weight=None


# ---------- P2: SVM-RFE + SVM tuyen tinh  (Guyon et al., 2002) ----------
class LinearSVMOvR:
    # SVM tuyen tinh qua kernel tinh san (nhanh khi n << p); one-vs-rest khi > 2 lop; co trong so w
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        Kmat = X @ X.T
        targets = [self.classes_[1]] if len(self.classes_) == 2 else list(self.classes_)
        W, b = [], []
        for c in targets:
            svc = SVC(kernel="precomputed", C=SVM_C).fit(Kmat, (y == c).astype(int))
            W.append(svc.dual_coef_[0] @ X[svc.support_])
            b.append(svc.intercept_[0])
        self.W, self.b = np.array(W), np.array(b)
        return self

    def predict(self, X):
        dec = X @ self.W.T + self.b
        if len(self.classes_) == 2:
            return self.classes_[(dec[:, 0] > 0).astype(int)]
        return self.classes_[dec.argmax(1)]


def svm_rfe_panels(X, y, k_list):
    # Loai mot nua so gene moi vong (Guyon et al.), dung dung tai moi k -> {k: (gene, SVM da fit tren k gene)}
    targets = sorted(k_list, reverse=True)
    active = np.arange(X.shape[1])
    out = {}
    while True:
        m = LinearSVMOvR().fit(X[:, active], y)
        if len(active) in targets:
            out[len(active)] = (active.copy(), m)
        if len(active) <= targets[-1]:
            break
        n_next = max(len(active) // 2, max(t for t in targets if t < len(active)))
        active = active[np.argsort(-(m.W ** 2).sum(0), kind="stable")[:n_next]]
    return out


# ---------- P3: Nearest shrunken centroids / PAM  (Tibshirani et al., 2002) ----------
class NSCStats:
    # Thong ke cua 1 fold; model_for_k(k) = PAM voi nguong Delta giu lai dung k gene
    def __init__(self, X, y):
        self.classes_ = np.unique(y)
        idx = np.searchsorted(self.classes_, y)
        n, K = len(y), len(self.classes_)
        nk = np.bincount(idx)
        self.xbar = X.mean(0)
        self.mu = np.stack([X[idx == j].mean(0) for j in range(K)])
        self.s = np.sqrt(((X - self.mu[idx]) ** 2).sum(0) / max(n - K, 1))     # do lech chuan chung trong lop
        self.s0 = np.median(self.s)
        self.mk = np.sqrt(1.0 / nk - 1.0 / n)
        self.d = (self.mu - self.xbar) / (self.mk[:, None] * (self.s + self.s0))
        self.score = np.abs(self.d).max(0)
        self.order = np.argsort(-self.score, kind="stable")
        self.logprior = np.log(nk / n)                                          # prior = ti le lop

    def model_for_k(self, k, genes=None):
        if genes is None:            # PAM: nguong Delta de con dung k gene co centroid khac trung binh chung
            sel = self.order[:k]
            delta = self.score[self.order[k]] if k < len(self.order) else 0.0
        else:                        # random-k: gene cho truoc, khong rut gon
            sel, delta = np.asarray(genes), 0.0
        d = self.d[:, sel]
        d_sh = np.sign(d) * np.maximum(np.abs(d) - delta, 0.0)                  # soft-threshold
        scale = self.s[sel] + self.s0
        return _NSCModel(self.classes_, sel, self.xbar[sel] + self.mk[:, None] * scale * d_sh,
                         scale ** 2, self.logprior)


class _NSCModel:
    def __init__(self, classes, sel, mu, scale2, logprior):
        self.classes_, self.sel, self.mu, self.scale2, self.logprior = classes, sel, mu, scale2, logprior

    def predict(self, X):
        Xs = X[:, self.sel]
        d = -(((Xs[:, None, :] - self.mu[None]) ** 2) / self.scale2).sum(2) + 2 * self.logprior
        return self.classes_[d.argmax(1)]


# ---------- 1 fold: moi pipeline, moi k ----------
def fold_predictions(Xtr, ytr, Xte, pipeline):
    # -> {k: du doan tren test}
    if pipeline == "ftest_lr":
        order = ftest_rank(Xtr, ytr)
        return {k: make_lr().fit(Xtr[:, order[:k]], ytr).predict(Xte[:, order[:k]]) for k in K_LIST}
    if pipeline == "svm_rfe":
        return {k: m.predict(Xte[:, g]) for k, (g, m) in svm_rfe_panels(Xtr, ytr, K_LIST).items()}
    if pipeline == "pam":
        st = NSCStats(Xtr, ytr)
        return {k: st.model_for_k(k).predict(Xte) for k in K_LIST}
    raise ValueError(pipeline)


def random_predictions(Xtr, ytr, Xte, genes, pipeline):
    # random-k: k gene ngau nhien + bo phan loai cua pipeline
    a, b = Xtr[:, genes], Xte[:, genes]
    if pipeline == "ftest_lr":
        return make_lr().fit(a, ytr).predict(b)
    if pipeline == "svm_rfe":
        return LinearSVMOvR().fit(a, ytr).predict(b)
    if pipeline == "pam":
        return NSCStats(a, ytr).model_for_k(None, genes=np.arange(len(genes))).predict(b)
    raise ValueError(pipeline)

## 4. Chạy một lần (nhãn thật hoặc một hoán vị)

`pid = -1` là nhãn thật; `pid = 0..999` là hoán vị. Nhãn của hoán vị `pid` cho nhiệm vụ `t` được xáo bằng seed
`(SEED, pid, t)` (theo bệnh nhân) → **tất định**: mọi phiên chạy, mọi pipeline dùng đúng cùng một bộ nhãn xáo.
Điểm của một lần chạy = BAcc trung bình qua 5 fold.

In [ ]:
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits

TASK_INDEX = {t: i for i, t in enumerate(["stage_all", "early_late", "stage_merge_34"])}


def permute_labels(y, groups, rng):
    # Xao theo benh nhan neu moi benh nhan chi co 1 nhan; nguoc lai xao theo mau
    s = pd.Series(y, index=groups)
    if (s.groupby(level=0).nunique() == 1).all():
        lab = s.groupby(level=0).first()
        mapping = dict(zip(lab.index, rng.permutation(lab.to_numpy())))
        return np.array([mapping[g] for g in groups])
    return rng.permutation(y)


def labels_for(pid, task, task_labels, groups):
    y = task_labels[task]
    if pid < 0:
        return y
    return permute_labels(y, groups, np.random.default_rng([SEED, pid, TASK_INDEX[task]]))


def run_pid(pid, pipelines, fold_data, folds, task_labels, groups):
    # -> list dong {pid, task, pipeline, k, bacc}; 1 tien trinh, BLAS 1 luong
    warnings.filterwarnings("ignore")
    rows = []
    with threadpool_limits(1):
        for task in task_labels:
            y = labels_for(pid, task, task_labels, groups)
            for p in pipelines:
                acc = {k: [] for k in K_LIST}
                for (Xtr, Xte), (tr, te) in zip(fold_data, folds):
                    for k, pred in fold_predictions(Xtr, y[tr], Xte, p).items():
                        acc[k].append(balanced_accuracy_score(y[te], pred))
                rows += [{"pid": pid, "task": task, "pipeline": p, "k": k, "bacc": float(np.mean(v))}
                         for k, v in acc.items()]
    return rows


def run_chunk(items, fold_data, folds, task_labels, groups):
    # items: [(pid, [pipeline,...])]
    return [r for pid, pipes in items for r in run_pid(pid, pipes, fold_data, folds, task_labels, groups)]

## 5. Đọc kết quả phiên trước (nếu có) và hàm lưu

In [ ]:
NULL_CSV = os.path.join(RESULT_DIR, "null_scores.csv")
RAND_CSV = os.path.join(RESULT_DIR, "random_k_scores.csv")
REAL_CSV = os.path.join(RESULT_DIR, "real_scores_per_fold.csv")
CONF_CSV = os.path.join(RESULT_DIR, "real_confusion.csv")
TIME_CSV = os.path.join(RESULT_DIR, "timing.csv")


DTYPES = {"pid": int, "rep": int, "k": int, "fold": int, "true": int, "pred": int, "count": int,
          "bacc": float, "sec_per_run_all_tasks": float}


def typed(d):
    return d.astype({c: t for c, t in DTYPES.items() if c in d.columns})


def load_prev(path, keys, cols):
    # Gop file cung ten tu RESUME_DIRS va RESULT_DIR, bo dong trung, ghi lai vao RESULT_DIR
    name = os.path.basename(path)
    srcs = [os.path.join(d, name) for d in RESUME_DIRS + [RESULT_DIR]]
    frames = [pd.read_csv(f) for f in srcs if os.path.isfile(f) and os.path.getsize(f) > 0]
    if not frames:
        return typed(pd.DataFrame(columns=cols))
    d = typed(pd.concat(frames, ignore_index=True).drop_duplicates(keys, keep="last"))
    d.to_csv(path, index=False)
    return d


def append_csv(rows, path):
    if rows:
        pd.DataFrame(rows).to_csv(path, mode="a", index=False, header=not os.path.exists(path))


null_df = load_prev(NULL_CSV, ["pid", "task", "pipeline", "k"], ["pid", "task", "pipeline", "k", "bacc"])
rand_df = load_prev(RAND_CSV, ["rep", "task", "pipeline", "k"], ["rep", "task", "pipeline", "k", "bacc"])
real_df = load_prev(REAL_CSV, ["task", "pipeline", "k", "fold"], ["task", "pipeline", "k", "fold", "bacc"])
conf_df = load_prev(CONF_CSV, ["task", "pipeline", "k", "true", "pred"], ["task", "pipeline", "k", "true", "pred", "count"])
time_df = load_prev(TIME_CSV, ["pipeline"], ["pipeline", "sec_per_run_all_tasks"])
print(f"Da co: {null_df.pid.nunique()} hoan vi | {rand_df.rep.nunique()} lan random-k | "
      f"nhan that: {sorted(real_df.pipeline.unique())}")


def hours_left():
    return TIME_BUDGET_H - (time.time() - T_START) / 3600

## 6. Nhãn thật (kèm ma trận nhầm lẫn, đo thời gian để ước lượng)

In [ ]:
from sklearn.metrics import confusion_matrix

real_rows, conf_rows, time_rows = [], [], []
for p in [p for p in PIPELINES if p not in set(real_df.pipeline)]:
    t0 = time.perf_counter()
    for task in TASKS:
        y = TASK_LABELS[task]
        labels = np.unique(y)
        pooled = {k: np.empty_like(y) for k in K_LIST}
        for f, ((Xtr, Xte), (tr, te)) in enumerate(zip(FOLD_DATA, FOLDS), 1):
            for k, pred in fold_predictions(Xtr, y[tr], Xte, p).items():
                pooled[k][te] = pred
                real_rows.append({"task": task, "pipeline": p, "k": k, "fold": f,
                                  "bacc": balanced_accuracy_score(y[te], pred)})
        for k in K_LIST:
            cm = confusion_matrix(y, pooled[k], labels=labels)
            conf_rows += [{"task": task, "pipeline": p, "k": k, "true": int(a), "pred": int(b), "count": int(cm[i, j])}
                          for i, a in enumerate(labels) for j, b in enumerate(labels)]
    sec = time.perf_counter() - t0
    time_rows.append({"pipeline": p, "sec_per_run_all_tasks": sec})
    print(f"{p:10s}: 1 lan chay (moi nhiem vu, moi k) = {sec:.1f}s")
append_csv(real_rows, REAL_CSV); append_csv(conf_rows, CONF_CSV); append_csv(time_rows, TIME_CSV)
real_df = typed(pd.read_csv(REAL_CSV)); time_df = typed(pd.read_csv(TIME_CSV))

real_mean = real_df.groupby(["task", "pipeline", "k"])["bacc"].mean().unstack("k").round(3)
display(real_mean)

# Uoc luong thoi gian (1 lan chay o tren dung BLAS nhieu luong -> uoc luong hoi lac quan)
done_null = null_df.groupby("pipeline").pid.nunique() if len(null_df) else pd.Series(dtype=int)
est = 0.0
for p in PIPELINES:
    sec = float(time_df.set_index("pipeline").loc[p, "sec_per_run_all_tasks"])
    left = N_PERMUTATIONS - int(done_null.get(p, 0))
    est += sec * (left + N_RANDOM_K * 0.5) / (N_WORKERS * 0.85)
print(f"Uoc luong con lai: ~{est / 3600:.1f} gio (ngan sach con {hours_left():.1f} gio)"
      + ("" if est / 3600 < hours_left() else " -> se dung an toan va can phien tiep theo (RESUME_DIRS)"))

## 7. Permutation test — 1000 hoán vị

Chạy theo lô; sau mỗi lô ghi thêm vào `null_scores.csv`. Nếu lô tiếp theo có thể vượt `TIME_BUDGET_H` thì dừng
an toàn — chạy phiên mới với `RESUME_DIRS` để tiếp tục.

In [ ]:
def pending_null():
    done = null_df.groupby("pipeline").pid.apply(set).to_dict() if len(null_df) else {}
    items = []
    for pid in range(N_PERMUTATIONS):
        pipes = [p for p in PIPELINES if pid not in done.get(p, set())]
        if pipes:
            items.append((pid, pipes))
    return items


items = pending_null()
batch = N_WORKERS * BATCH_PER_WORKER
print(f"Can chay {len(items)} hoan vi (lo {batch})")
stopped = False
for s in range(0, len(items), batch):
    lot = items[s:s + batch]
    t0 = time.time()
    chunks = [lot[i::N_WORKERS] for i in range(N_WORKERS) if lot[i::N_WORKERS]]
    res = Parallel(n_jobs=N_JOBS)(delayed(run_chunk)(c, FOLD_DATA, FOLDS, TASK_LABELS, groups) for c in chunks)
    rows = [r for part in res for r in part]
    append_csv(rows, NULL_CSV)
    null_df = typed(pd.concat([null_df, pd.DataFrame(rows)], ignore_index=True))
    lot_h = (time.time() - t0) / 3600
    n_done = s + len(lot)
    print(f"  {n_done}/{len(items)} hoan vi | lo {lot_h * 60:.1f} phut | con ~{lot_h * (len(items) - n_done) / len(lot):.1f} gio"
          f" | ngan sach con {hours_left():.1f} gio", flush=True)
    if n_done < len(items) and hours_left() < 1.5 * lot_h + 0.25:
        stopped = True
        print("DUNG AN TOAN: sap het ngan sach thoi gian. Chay phien moi voi RESUME_DIRS de tiep tuc.")
        break
print("Permutation:", "CHUA XONG" if stopped else "xong")

## 8. Baseline random-k

Với mỗi lần lặp, mỗi nhiệm vụ, mỗi fold và mỗi k: chọn ngẫu nhiên k gene (seed tất định), rồi huấn luyện **cùng bộ
phân loại** của từng pipeline trên đúng các gene đó. Chỉ chạy khi permutation đã xong.

In [ ]:
def run_random_rep(rep, pipelines, fold_data, folds, task_labels):
    warnings.filterwarnings("ignore")
    rows = []
    with threadpool_limits(1):
        for task, y in task_labels.items():
            rng = np.random.default_rng([SEED, 7, rep, TASK_INDEX[task]])
            acc = {(p, k): [] for p in pipelines for k in K_LIST}
            for (Xtr, Xte), (tr, te) in zip(fold_data, folds):
                for k in K_LIST:
                    genes = rng.choice(Xtr.shape[1], size=k, replace=False)   # cung gene cho moi pipeline
                    for p in pipelines:
                        acc[(p, k)].append(balanced_accuracy_score(y[te], random_predictions(Xtr, y[tr], Xte, genes, p)))
            rows += [{"rep": rep, "task": task, "pipeline": p, "k": k, "bacc": float(np.mean(v))}
                     for (p, k), v in acc.items()]
    return rows


def run_random_chunk(items, fold_data, folds, task_labels):
    return [r for rep, pipes in items for r in run_random_rep(rep, pipes, fold_data, folds, task_labels)]


if not stopped:
    done = rand_df.groupby("pipeline").rep.apply(set).to_dict() if len(rand_df) else {}
    items = [(r, [p for p in PIPELINES if r not in done.get(p, set())]) for r in range(N_RANDOM_K)]
    items = [it for it in items if it[1]]
    print(f"Can chay {len(items)} lan random-k")
    for s in range(0, len(items), batch):
        lot = items[s:s + batch]
        t0 = time.time()
        chunks = [lot[i::N_WORKERS] for i in range(N_WORKERS) if lot[i::N_WORKERS]]
        res = Parallel(n_jobs=N_JOBS)(delayed(run_random_chunk)(c, FOLD_DATA, FOLDS, TASK_LABELS) for c in chunks)
        rows = [r for part in res for r in part]
        append_csv(rows, RAND_CSV)
        rand_df = typed(pd.concat([rand_df, pd.DataFrame(rows)], ignore_index=True))
        lot_h = (time.time() - t0) / 3600
        print(f"  {s + len(lot)}/{len(items)} | lo {lot_h * 60:.1f} phut | ngan sach con {hours_left():.1f} gio", flush=True)
        if s + len(lot) < len(items) and hours_left() < 1.5 * lot_h + 0.25:
            stopped = True
            print("DUNG AN TOAN trong random-k. Chay phien moi voi RESUME_DIRS de tiep tuc.")
            break

## 9. Tổng hợp: p gốc, Holm, Bonferroni, Westfall–Young maxT; quyết định RQ0

- **p gốc:** p = (#{hoán vị: BAcc ≥ BAcc thật} + 1) / (N + 1) (Phipson & Smyth, 2010).
- **Họ kiểm định:** 3 pipeline × 5 k = 15 kiểm định cho mỗi (bộ dữ liệu, nhiệm vụ). Holm (1979) và Bonferroni
  hiệu chỉnh trong họ này. Với N = 1000, p nhỏ nhất = 1/1001 ≈ 0,001 < 0,05/15 ≈ 0,0033 nên vẫn có thể đạt ý nghĩa.
- **maxT (tham khảo):** chuẩn hoá điểm của mỗi kiểm định theo phân phối null của nó, lấy max qua 15 kiểm định ở
  mỗi hoán vị → p hiệu chỉnh ít bảo thủ hơn khi các kiểm định tương quan (Westfall & Young, 1993).
- **So với random-k:** p_vs_random = (#{lần random ≥ thật} + 1)/(N_random + 1) (không hiệu chỉnh, tiêu chí phụ) và
  mức chênh `gain = BAcc thật − TB random-k`.
- **Quyết định (đặt trước):** một kiểm định *có tín hiệu* nếu `p_holm < 0,05`; *chọn gen có ích* nếu thêm
  `p_vs_random < 0,05` **và** `gain ≥ MIN_GAIN`. Nhiệm vụ **đạt RQ0** nếu có ít nhất một kiểm định "có ích".

In [ ]:
def holm(p):
    p = np.asarray(p, float); m = len(p); o = np.argsort(p)
    adj = np.maximum.accumulate((m - np.arange(m)) * p[o]).clip(max=1)
    out = np.empty(m); out[o] = adj
    return out


real_mean = real_df.groupby(["task", "pipeline", "k"])["bacc"].mean()
rows = []
for task in TASKS:
    nt = null_df[null_df.task == task]
    for p in ALL_PIPELINES:
        for k in K_LIST:
            if (task, p, k) not in real_mean.index:
                continue
            null = nt[(nt.pipeline == p) & (nt.k == k)]["bacc"].to_numpy()
            if len(null) == 0:
                continue
            real = real_mean[(task, p, k)]
            rk = rand_df[(rand_df.task == task) & (rand_df.pipeline == p) & (rand_df.k == k)]["bacc"].to_numpy()
            rows.append({
                "dataset": DATASET_NAME, "task": task, "pipeline": p, "k": k, "chance": TASK_CHANCE[task],
                "bacc_real": real,
                "bacc_real_sd_folds": real_df.query("task == @task and pipeline == @p and k == @k")["bacc"].std(ddof=1),
                "n_perm": len(null), "null_mean": null.mean(), "null_sd": null.std(ddof=1),
                "null_q95": np.quantile(null, 0.95),
                "p_raw": (np.sum(null >= real) + 1) / (len(null) + 1),
                "n_random": len(rk),
                "random_mean": rk.mean() if len(rk) else np.nan,
                "random_q95": np.quantile(rk, 0.95) if len(rk) else np.nan,
                "p_vs_random": (np.sum(rk >= real) + 1) / (len(rk) + 1) if len(rk) else np.nan,
                "gain_vs_null": real - null.mean(),
                "gain_vs_random": real - rk.mean() if len(rk) else np.nan,
            })
summary = pd.DataFrame(rows)

# Hieu chinh trong ho (bo, nhiem vu); maxT tren cac hoan vi chung cua ca ho
family_size = len(ALL_PIPELINES) * len(K_LIST)
adj_parts = []
for task, g in summary.groupby("task"):
    g = g.copy()
    g["family_size_expected"] = family_size
    g["family_complete"] = (len(g) == family_size) and (g.n_perm.min() >= N_PERMUTATIONS)
    g["p_holm"] = holm(g.p_raw)
    g["p_bonferroni"] = (g.p_raw * len(g)).clip(upper=1)
    piv = null_df[null_df.task == task].pivot_table(index="pid", columns=["pipeline", "k"], values="bacc").dropna()
    keys = list(zip(g.pipeline, g.k))
    mu, sd = piv[keys].mean(), piv[keys].std(ddof=1).replace(0, np.nan)
    Z = (piv[keys] - mu) / sd
    z_real = (g.bacc_real.to_numpy() - mu.to_numpy()) / sd.to_numpy()
    zmax = Z.max(axis=1).to_numpy()
    g["p_maxT"] = [(np.sum(zmax >= z) + 1) / (len(zmax) + 1) for z in z_real]
    adj_parts.append(g)
summary = pd.concat(adj_parts, ignore_index=True)
summary["signal"] = summary.p_holm < ALPHA
summary["useful"] = summary.signal & (summary.p_vs_random < ALPHA) & (summary.gain_vs_random >= MIN_GAIN)
summary.to_csv(os.path.join(RESULT_DIR, "summary.csv"), index=False)

decision = summary.groupby("task").agg(
    n_tests=("p_raw", "size"), family_complete=("family_complete", "first"),
    best_bacc=("bacc_real", "max"), min_p_raw=("p_raw", "min"), min_p_holm=("p_holm", "min"),
    n_signal=("signal", "sum"), n_useful=("useful", "sum")).reset_index()
decision["chance"] = decision.task.map(TASK_CHANCE)
decision["verdict"] = np.where(decision.n_useful > 0, "DAT: co tin hieu va chon gen co ich",
                      np.where(decision.n_signal > 0, "Co tin hieu nhung chon gen KHONG hon random-k",
                               "KHONG co bang chung ve tin hieu"))
decision.to_csv(os.path.join(RESULT_DIR, "decision.csv"), index=False)
if not summary.family_complete.all():
    print("CANH BAO: ho kiem dinh chua du (thieu pipeline hoac chua du hoan vi) -> p hieu chinh CHUA phai ket qua cuoi.")
display(decision)
cols = ["task", "pipeline", "k", "bacc_real", "null_mean", "p_raw", "p_holm", "p_bonferroni", "p_maxT",
        "random_mean", "p_vs_random", "gain_vs_random", "signal", "useful"]
display(summary[cols].round(4))

In [ ]:
import matplotlib.pyplot as plt

PIPE_NAMES = {"ftest_lr": "F-test + LR", "svm_rfe": "SVM-RFE + SVM", "pam": "PAM"}
fig, axes = plt.subplots(len(TASKS), len(ALL_PIPELINES), figsize=(4 * len(ALL_PIPELINES), 3.2 * len(TASKS)),
                         squeeze=False, sharey="row")
for i, task in enumerate(TASKS):
    for j, p in enumerate(ALL_PIPELINES):
        ax = axes[i][j]
        g = summary[(summary.task == task) & (summary.pipeline == p)].sort_values("k")
        if len(g):
            x = np.arange(len(g))
            ax.fill_between(x, g.null_mean - 1.96 * g.null_sd, g.null_mean + 1.96 * g.null_sd, alpha=0.25,
                            label="Null 95%")
            ax.plot(x, g.random_mean, "s--", label="Random-k TB")
            ax.plot(x, g.bacc_real, "o-", color="k", label="Nhãn thật")
            for xi, (_, r) in zip(x, g.iterrows()):
                if r.signal:
                    ax.annotate("*", (xi, r.bacc_real), ha="center", va="bottom", fontsize=14)
            ax.set_xticks(x); ax.set_xticklabels(g.k)
        ax.axhline(TASK_CHANCE[task], color="gray", ls=":", lw=1)
        ax.set_title(f"{task} · {PIPE_NAMES[p]}", fontsize=9)
        if j == 0:
            ax.set_ylabel("BAcc (TB 5 fold)")
        if i == len(TASKS) - 1:
            ax.set_xlabel("k (số gene)")
axes[0][0].legend(fontsize=7)
fig.suptitle(f"{DATASET_NAME}: BAcc thật vs null vs random-k (* = p_holm < {ALPHA})")
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, "perm_test_v2_bacc.png"), dpi=150)
plt.show()

fig, axes = plt.subplots(1, len(TASKS), figsize=(4.5 * len(TASKS), 3.2), squeeze=False)
for ax, task in zip(axes[0], TASKS):
    h = summary[summary.task == task].pivot_table(index="pipeline", columns="k", values="p_holm")
    h = h.reindex([p for p in ALL_PIPELINES if p in h.index])
    im = ax.imshow(-np.log10(h.values), cmap="viridis", vmin=0, vmax=3)
    ax.set_xticks(range(h.shape[1])); ax.set_xticklabels(h.columns)
    ax.set_yticks(range(h.shape[0])); ax.set_yticklabels([PIPE_NAMES[p] for p in h.index])
    for a in range(h.shape[0]):
        for b in range(h.shape[1]):
            ax.text(b, a, f"{h.values[a, b]:.3f}", ha="center", va="center", fontsize=7,
                    color="w" if h.values[a, b] > 0.01 else "k")
    ax.set_title(f"{task}: p Holm"); ax.set_xlabel("k")
fig.colorbar(im, ax=axes[0].tolist(), label="-log10(p_holm)")
plt.savefig(os.path.join(RESULT_DIR, "perm_test_v2_pvalues.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import shutil, sklearn
with open(os.path.join(RESULT_DIR, "config.json"), "w") as f:
    json.dump({"dataset": DATASET_NAME, "data_dir": DATA_DIR, "pipelines_this_run": PIPELINES,
               "all_pipelines": ALL_PIPELINES, "k_list": K_LIST, "n_permutations": N_PERMUTATIONS,
               "n_random_k": N_RANDOM_K, "n_splits": N_SPLITS, "seed": SEED, "lr_c": LR_C, "svm_c": SVM_C,
               "class_weight": None, "alpha": ALPHA, "min_gain": MIN_GAIN, "tasks": TASKS,
               "n_tumour_samples": int(len(y_stage)), "n_genes": int(X.shape[1]),
               "n_patients_with_multiple_samples": n_dup, "scaling": "none (raw log2)",
               "resume_dirs": RESUME_DIRS, "stopped_early": stopped, "cpu_count": os.cpu_count(),
               "sklearn": sklearn.__version__, "numpy": np.__version__,
               "elapsed_hours": (time.time() - T_START) / 3600}, f, indent=1)
zip_path = shutil.make_archive(f"/kaggle/working/perm_test_v2_{DATASET_NAME}", "zip", RESULT_DIR)
print("Da nen:", zip_path, f"| tong thoi gian {(time.time() - T_START) / 3600:.2f} gio")
for fn in sorted(os.listdir(RESULT_DIR)):
    print(" ", fn)